<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week11_Agent_Simulations_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 11 Practical (PH):** Multi-agent simulation — boids, ant foraging and message-passing network search

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO2, LO5

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — Boids on a unit torus | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part B — Ant foraging (stigmergy) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — Message-passing search in a network (communicating agents) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part D — Gossip (consensus) among communicating agents | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | explain **emergence**: global order from local rules | Understand | LO2 |
| 2 | implement **boids** and measure flocking with an **order parameter** | Apply / Evaluate | LO2 |
| 3 | implement **ant-colony** foraging with pheromone and test it against a no-communication baseline | Apply / Analyse | LO5 |
| 4 | compare **flooding vs random-walk** search in a network by *message cost* | Analyse | LO5 |
| 5 | run a **gossip** (consensus) protocol among communicating agents | Apply | LO2 |

**Emergence.** No boid knows the flock's shape; no ant knows the shortest path. Each agent follows *local* rules and *indirect or direct communication* (stigmergy, messages) produces global behaviour. Because outcomes are random, **every claim needs several seeds and an interval** — one pretty animation is an anecdote, not evidence.

**Order parameter for flocking.** With unit headings $\mathbf u_i=\mathbf v_i/\lVert\mathbf v_i\rVert$,
$$\varphi=\Bigl\lVert \tfrac1N\sum_{i=1}^N\mathbf u_i\Bigr\rVert\in[0,1],$$
$\varphi\approx 0$ for random headings and $\varphi\to1$ for a fully aligned flock.

**Do Now (3 min).** Name one real-world system (traffic, markets, mobile-money fraud rings, WhatsApp rumours) where global behaviour emerges from local interaction.

In [ ]:
#@title 📦 Imports
# Plotting library
import matplotlib.pyplot as plt
# SciPy statistics
from scipy import stats
# networkx builds and analyses graphs (the communication network)
import networkx as nx

## Part A — Boids on a unit torus
Each boid $i$ looks at neighbours within radius $R$ and steers toward a weighted sum of three unit-vector rules:

* **alignment** — the mean heading of neighbours (including itself),
* **cohesion** — toward the neighbours' centre of mass,
* **separation** — away from neighbours closer than $r_s$.

The new heading is $\operatorname{normalise}(w_a\mathbf a+w_c\mathbf c+w_s\mathbf s)$ plus angular noise $\eta\sim\mathcal N(0,\sigma^2)$; speed is constant. **Task:** implement the alignment term.

### 🔍 Code walkthrough — three local rules, one global behaviour
No boid knows where the flock is going. Each one looks only at neighbours within radius $R$ and blends three steering rules: **alignment** (match their heading), **cohesion** (move towards their centre) and **separation** (avoid crowding). The order parameter $\varphi$ measures the result: 1 means everyone points the same way, near 0 means chaos. Your job is the alignment rule; the experiment then tests whether it *causes* order.

In [ ]:
#@title 🐦 Boid simulation (TODO: alignment_term)
# Normalise vectors to length 1 (safe when the length is 0)
def unit(v):
    # Vector lengths; divide, avoiding division by zero
    n = np.linalg.norm(v, axis=-1, keepdims=True); return v / np.where(n < 1e-12, 1.0, n)

# Who is within radius R of whom
def neighbours(pos, R):
    # Displacements on a wrap-around (torus) world
    d = pos[:, None, :] - pos[None, :, :]; d -= np.round(d)               # periodic (torus) displacement
    # Distances and the neighbour mask
    dist = np.linalg.norm(d, axis=-1); return d, dist, (dist < R)

def alignment_term(vel, mask):
    """TODO(Student): for each boid, the mean of the UNIT headings of all boids j with mask[i, j] True (mask includes i itself).
       Return an (N, 2) array (not necessarily unit length)."""
    # Placeholder: returns the normalised velocities (replace with the alignment rule)
    return unit(vel)

def order_parameter(vel):
    """φ = ‖mean of unit headings‖."""
    # Placeholder order parameter (replace with the mean heading length)
    return float(np.linalg.norm(unit(vel).mean(0)))

# Boids simulation: alignment (wa), cohesion (wc), separation (ws)
def run_boids(wa=1.0, wc=0.3, ws=0.5, N=60, steps=250, R=0.2, rs_=0.04, speed=0.015, noise=0.15, seed=42, record=False):
    # Random positions and headings; velocities from angles
    rs = np.random.default_rng(seed); pos = rs.random((N, 2)); th = rs.uniform(0, 2 * np.pi, N); vel = np.c_[np.cos(th), np.sin(th)]
    # Order parameter history
    phis = []
    # Simulate each time step
    for t in range(steps):
        # Find neighbours
        d, dist, m = neighbours(pos, R)
        # Alignment: average heading of neighbours
        a = alignment_term(vel, m)
        # Cohesion: steer towards the neighbours' centre
        cnt = m.sum(1, keepdims=True); c = unit(-(d * m[:, :, None]).sum(1) / cnt)                      # toward centre of neighbours
        # Separation: steer away from very close boids
        close = (dist < rs_) & ~np.eye(N, dtype=bool); s = unit((d * close[:, :, None]).sum(1))        # away from very close boids
        # Combine the three steering rules and add heading noise
        h = unit(wa * a + wc * c + ws * s); ang = np.arctan2(h[:, 1], h[:, 0]) + rs.normal(0, noise, N)
        # Move the boids, wrapping around the world; record the order
        vel = np.c_[np.cos(ang), np.sin(ang)]; pos = (pos + speed * vel) % 1.0; phis.append(order_parameter(vel))
    # Return the history and the final state
    return np.array(phis), pos, vel

In [ ]:
#@title ✅ Auto-check
# Tiny test: 3 boids and who sees whom
mk = np.array([[True, True, False], [True, True, True], [False, True, True]]); vv = np.array([[2.0, 0], [0, 3.0], [-1.0, 0]])
# Check your alignment on the test case
A = alignment_term(vv, mk)
check("alignment of boid 0 = mean of unit headings of {0,1} = (0.5, 0.5)", lambda: np.allclose(A[0], [0.5, 0.5]))
check("alignment of boid 1 = mean of {0,1,2} = (0, 1/3)",               lambda: np.allclose(A[1], [0.0, 1/3]))
check("order_parameter: all aligned → 1",                                 lambda: abs(order_parameter(np.tile([1.0, 0], (10, 1))) - 1) < 1e-9)
check("order_parameter: opposite headings → 0",                           lambda: order_parameter(np.array([[1.0, 0], [-1.0, 0]])) < 1e-9)

### Does alignment *cause* order? A controlled experiment
We compare $w_a=0$ (no alignment) against $w_a=1$ over **15 seeds**, measuring the final order parameter (mean of the last 50 steps). The groups use the same seeds → paired design; we use the Wilcoxon signed-rank test.

In [ ]:
#@title 🧪 15 seeds × 2 conditions
# Results by alignment weight
res = {}
# Compare no alignment (0) with alignment (1)
for wa in (0.0, 1.0):
    # 15 seeded runs; average the last 50 steps
    res[wa] = np.array([run_boids(wa=wa, seed=s)[0][-50:].mean() for s in range(15)])
    # Mean, SD and 95% CI
    m, s, lo, hi = mean_ci(res[wa]); print(f"w_a = {wa}: φ = {m:.3f} ± {s:.3f}  (95% CI [{lo:.3f}, {hi:.3f}])")
# One-sided paired Wilcoxon: alignment raises the order
w = stats.wilcoxon(res[1.0], res[0.0], alternative="greater"); print(f"paired Wilcoxon: W = {w.statistic:.0f}, p = {w.pvalue:.2e}")
# One run with and one without alignment for the plot
phis, pos, vel = run_boids(wa=1.0, seed=42); phis0, pos0, vel0 = run_boids(wa=0.0, seed=42)
# Three panels
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
# Order parameter over time
ax[0].plot(phis0, label="$w_a=0$"); ax[0].plot(phis, label="$w_a=1$"); ax[0].set_xlabel("step"); ax[0].set_ylabel("order parameter φ"); ax[0].legend()
# Snapshots of headings
for a_, (p_, v_, t_) in zip(ax[1:], [(pos0, vel0, "no alignment"), (pos, vel, "with alignment")]): a_.quiver(p_[:, 0], p_[:, 1], v_[:, 0], v_[:, 1], scale=25); a_.set_title(t_); a_.set_xlim(0, 1); a_.set_ylim(0, 1)
# Display
plt.tight_layout(); plt.show()
check("alignment produces high order (φ > 0.8)",  lambda: res[1.0].mean() > 0.8)
check("without alignment order stays low (φ < 0.5)", lambda: res[0.0].mean() < 0.5)
check("the difference is significant (p < 0.01)",  lambda: w.pvalue < 0.01)

## Part B — Ant foraging (stigmergy)
Ants live on an $n\times n$ grid. The **nest** is on the left, **food** on the right. A *searching* ant moves to a neighbouring cell with probability
$$P(j\mid i)\;\propto\;(\varepsilon+\tau_j)^{\alpha},$$
where $\tau$ is the pheromone level and $\varepsilon>0$ keeps exploration alive. A searching ant never steps straight back to the cell it just left, and it **erases loops** from its remembered path. On finding food it turns **returning**, retraces that path to the nest, and deposits $Q/L$ on each cell it crosses ($L$ = path length, so short paths get stronger trails). Pheromone evaporates each step: $\tau\leftarrow(1-\rho)\tau$. Baseline: $\alpha=0$ (non-backtracking random walk, no communication). **Task:** implement `choose_next`.

In [ ]:
#@title 🐜 Ant colony (TODO: choose_next)
def choose_next(tau_nb, alpha, eps, rs):
    """TODO(Student): tau_nb = array of pheromone values of the candidate neighbour cells.
       Return an index sampled with probability ∝ (eps + tau)^alpha (use rs.choice)."""
    # Placeholder: choose the first neighbour (replace)
    return int(rs.integers(len(tau_nb)))

# Ant colony on a grid: pheromone guides the route to food
def run_ants(alpha=1.5, n=15, n_ants=20, T=1500, rho=0.01, Q=5.0, eps=0.1, seed=42):
    # Seeded generator, nest and food positions, empty pheromone grid
    rs = np.random.default_rng(seed); nest, food = (n // 2, 1), (n // 2, n - 2); tau = np.zeros((n, n))
    # Create the ants and counters
    ants = [{"pos": nest, "path": [nest], "mode": "search", "dep": 0.0} for _ in range(n_ants)]; delivered = 0; hist = []
    # Simulate each time step
    for t in range(T):
        # For each ant
        for a in ants:
            # Searching ant: pick the next cell
            if a["mode"] == "search":
                # Neighbouring cells
                r, c = a["pos"]; nb_ = [(r + dr, c + dc) for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)) if 0 <= r + dr < n and 0 <= c + dc < n]
                # Previous cell (to avoid stepping back)
                prev = a["path"][-2] if len(a["path"]) > 1 else None
                # Candidate moves without immediate backtracking
                nb2 = [q for q in nb_ if q != prev] or nb_                        # no immediate backtracking
                # Choose using pheromone; move
                j = choose_next(np.array([tau[q] for q in nb2]), alpha, eps, rs); p = nb2[j]; a["pos"] = p
                # Loop erasure keeps the path free of loops
                if p in a["path"]: a["path"] = a["path"][:a["path"].index(p) + 1]    # loop erasure: keep the path loop-free
                # Otherwise extend the path
                else: a["path"].append(p)
                # Reached food: switch to returning and set deposit amount (shorter path = more)
                if p == food: a["mode"] = "return"; a["dep"] = Q / len(a["path"])   # shorter path → more pheromone per cell
            # Returning ant
            else:
                # Retrace one step and drop pheromone
                a["path"].pop(); a["pos"] = a["path"][-1]; tau[a["pos"]] += a["dep"]
                # At the nest: count delivery and search again
                if a["pos"] == nest: delivered += 1; a["mode"] = "search"; a["path"] = [nest]
        # Pheromone evaporates; record deliveries
        tau *= (1 - rho); hist.append(delivered)
    # Return totals and the pheromone map
    return delivered, np.array(hist), tau

In [ ]:
#@title ✅ Auto-check: sampling distribution of choose_next
# Test the rule over 4000 draws
rs_t = np.random.default_rng(0); draws = [choose_next(np.array([0.0, 0.9]), 1.0, 0.1, rs_t) for _ in range(4000)]
# Fraction choosing the pheromone cell (expected about 0.909)
p1 = np.mean(np.array(draws) == 1)                                        # expected (0.1+0.9)/(0.1+1.0) = 0.909
check("probability ∝ (ε+τ)^α : P(cell 2) ≈ 0.909",  lambda: abs(p1 - 1.0 / 1.1) < 0.03)
# With alpha = 0 the choice must be uniform
d0 = [choose_next(np.array([0.0, 5.0]), 0.0, 0.1, rs_t) for _ in range(4000)]
check("α = 0 ignores pheromone: P(cell 2) ≈ 0.5",   lambda: abs(np.mean(np.array(d0) == 1) - 0.5) < 0.03)

In [ ]:
#@title 🧪 Pheromone vs no communication: 12 seeds
# Run 12 seeds for both alpha values
D = {a: np.array([run_ants(alpha=a, seed=s)[0] for s in range(12)]) for a in (0.0, 1.5)}
# Print deliveries with CI
for a, v in D.items(): m, s, lo, hi = mean_ci(v); print(f"α = {a}: food delivered in 1500 steps = {m:.1f} ± {s:.1f}  (95% CI [{lo:.1f}, {hi:.1f}])")
# One-sided paired Wilcoxon
w = stats.wilcoxon(D[1.5], D[0.0], alternative="greater"); print(f"paired Wilcoxon: W = {w.statistic:.0f}, p = {w.pvalue:.4f}")
# One run each for the plots
_, h0, _ = run_ants(alpha=0.0, seed=42); _, h1, tau = run_ants(alpha=1.5, seed=42)
# Plot deliveries over time
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2)); ax[0].plot(h0, label="α=0 (random)"); ax[0].plot(h1, label="α=1.5 (pheromone)"); ax[0].set_xlabel("step"); ax[0].set_ylabel("cumulative food delivered"); ax[0].legend()
# Show the pheromone field
ax[1].imshow(tau, cmap="magma"); ax[1].set_title("pheromone field at T=1500"); plt.tight_layout(); plt.show()
check("pheromone colony delivers more food on average", lambda: D[1.5].mean() > D[0.0].mean())
check("improvement is statistically significant (p < 0.05)", lambda: w.pvalue < 0.05)

**Discuss.** Which parameter — $\alpha$ (greed), $\rho$ (evaporation) or $\varepsilon$ (exploration) — would you tune if the food moves? Too little evaporation *locks in* a stale trail (premature convergence); too much destroys the memory. This is the explore–exploit trade-off again (Week 4: heuristic search).

## Part C — Message-passing search in a network (communicating agents)
$N$ nodes (agents) hold local data; one node must find the **resource holder** *without a global map*. Two protocols:

* **Flooding with TTL.** The query is forwarded to all neighbours (except the sender) once per node. Fast, but the number of messages grows with the network.
* **Random walk.** A single query hops to a random neighbour. Cheap per step, but slow and unbounded.

Cost = **messages sent**; also record **rounds** (parallel time). **Task:** implement `flood`.

In [ ]:
#@title 📨 flood (TODO) and random_walk
def flood(G, src, target, ttl=99):
    """TODO(Student): synchronous rounds. frontier = {src: None (no sender)}; seen = {src}.
    Each round every frontier node sends the query to all neighbours except its sender (count one message per transmission,
    including transmissions to already-seen nodes); nodes that have not seen the query become the next frontier.
    Stop at the end of the round in which `target` receives the query, or when the frontier is empty or ttl rounds elapse.
    Return (found: bool, messages: int, rounds: int)."""
    # Placeholder result (replace)
    return (False, 0, 0)

# Random-walk search on a network
def random_walk(G, src, target, rs, max_steps=5000):
    # Current node and step count
    cur, steps = src, 0
    # Walk until the target is found or the cap is hit
    while cur != target and steps < max_steps:
        # Move to a random neighbour
        cur = rs.choice(list(G.neighbors(cur))); steps += 1
    # Return success and steps
    return cur == target, steps

In [ ]:
#@title ✅ Auto-check
# Small test graphs
P5 = nx.path_graph(5); K4 = nx.complete_graph(4)
check("path 0–4: found, 4 messages, 4 rounds",                 lambda: flood(P5, 0, 4) == (True, 4, 4))
check("K4, target 3: found in 1 round with 3 messages",        lambda: flood(K4, 0, 3) == (True, 3, 1))
check("TTL = 2 on the path cannot reach node 4",               lambda: flood(P5, 0, 4, ttl=2)[0] is False)
check("src = target costs nothing",                            lambda: flood(P5, 2, 2) == (True, 0, 0))

In [ ]:
#@title 🧪 30 random networks (Watts–Strogatz small-world, n = 200)
# Seeded generator; results
rs = np.random.default_rng(42); R = []
# 30 random small-world networks
for k in range(30):
    # Build a network; random source and target
    G = nx.connected_watts_strogatz_graph(200, 6, 0.1, seed=int(rs.integers(1e9))); src, tgt = rs.choice(200, 2, replace=False)
    # Run both methods and store results
    f = flood(G, int(src), int(tgt)); r = random_walk(G, int(src), int(tgt), rs); R.append((f[1], f[2], r[1], f[0], r[0]))
# Print success rates
R = np.array(R, float); print("success: flood", R[:, 3].mean(), " random walk", R[:, 4].mean())
# For each cost measure
for name, col in [("flooding messages", 0), ("flooding rounds", 1), ("random-walk messages (=rounds)", 2)]:
    # Mean, SD and CI
    m, s, lo, hi = mean_ci(R[:, col]); print(f"{name:<32} {m:8.1f} ± {s:7.1f}  (95% CI [{lo:.1f}, {hi:.1f}])")
# Paired Wilcoxon on message counts
w = stats.wilcoxon(R[:, 0], R[:, 2], alternative="two-sided"); print(f"messages: flood vs random walk, paired Wilcoxon p = {w.pvalue:.3g}")
# Compare time versus message cost
print(f"time: flooding finishes in {R[:,1].mean():.1f} rounds vs {R[:,2].mean():.0f} for the walk → trade-off: speed vs message cost")
check("flooding always finds the target in a connected graph", lambda: R[:, 3].all())
check("flooding is much faster (rounds) than the random walk",  lambda: R[:, 1].mean() * 5 < R[:, 2].mean())

## Part D — Gossip (consensus) among communicating agents
Each agent holds a number (e.g. a local sensor reading). Periodically it averages with a random neighbour: $x_i,x_j\leftarrow\tfrac12(x_i+x_j)$. Pairwise averaging **preserves the global mean** and drives the variance to zero on a connected graph — a decentralised way to compute an average with no coordinator.

In [ ]:
#@title 🗣️ Pairwise gossip
# Gossip averaging: network, initial values, mean and variance
rs = np.random.default_rng(42); G = nx.connected_watts_strogatz_graph(50, 4, 0.2, seed=42); x = rs.normal(30, 5, 50); mu0 = x.mean(); var = [x.var()]
# List of links
edges = list(G.edges())
# 3000 gossip steps
for t in range(3000):
    # Pick a random link; both nodes take the average
    i, j = edges[rs.integers(len(edges))]; x[i] = x[j] = (x[i] + x[j]) / 2
    # Record the variance every 50 steps
    if t % 50 == 0: var.append(x.var())
# Mean is unchanged; variance falls
print(f"mean before = {mu0:.4f}, after = {x.mean():.4f}; variance {var[0]:.3f} → {var[-1]:.2e}")
# Plot variance on a log axis
plt.semilogy(var); plt.xlabel("×50 gossip steps"); plt.ylabel("variance"); plt.show()
check("global mean is preserved",          lambda: abs(x.mean() - mu0) < 1e-9)
check("agents reach consensus (var < 1e-3)", lambda: var[-1] < 1e-3)

## CA 3 (Week 11) — marking scheme (10 marks, 30-minute in-class practical)
| Item | Marks |
|---|---|
| Correct implementation of one TODO from this lab (alignment, `choose_next`, or `flood`) with passing auto-checks | 4 |
| An experiment with ≥ 10 seeds, mean ± sd / CI and a named test | 3 |
| One-paragraph interpretation (emergence, trade-offs) | 2 |
| Reproducibility (seed, logging, runs top-to-bottom) | 1 |

## 📝 CA3 deliverables (10 marks) — submit this notebook + a 1-page PDF
| Criterion | Marks |
|---|---|
| `alignment_term`, `choose_next`, `flood` correct (all auto-checks ✅) | 3 |
| Experiments: ≥ 12 seeds per condition, mean ± sd, 95% CI, Wilcoxon *p* and one sentence of interpretation | 3 |
| Comparison of **stigmergy, flooding and gossip**: what is communicated, cost, robustness | 2 |
| Reflection: where would a swarm or gossip protocol be a *bad* idea, and why? | 2 |

## Reflection
- Boids and ants both show emergence, but only ants *modify the environment*. Why does that matter for memory and for the explore–exploit trade-off?
- Flooding vs random walk: which would you use in a Nigerian rural mesh network with intermittent power and expensive data? Justify with the message-cost numbers.

```text
Write your reflection here…
```

## Extensions
- Add an **obstacle** to the ant grid and test whether pheromone finds a detour.
- Add a **predator** repulsion rule to the boids and track φ.
- Compare **k-walker** random walks ($k=1,4,16$) in Part C on the same graphs.

In [ ]:
checks_summary()  # how many auto-checks have you passed?